# Análise de Churn de Clientes Bancários

Este notebook realiza uma análise exploratória de dados de clientes bancários com foco em entender padrões relacionados ao churn.

## Objetivo

Analisar o perfil dos clientes do banco, comparar clientes que cancelaram e não cancelaram o serviço, e identificar características associadas ao churn.

## Perguntas de Negócio

1. Qual é o perfil geral dos clientes do banco?
2. Quais diferenças aparecem entre clientes que cancelaram e clientes que permaneceram?
3. O comportamento dos clientes muda entre Alemanha, França e Espanha?
4. Quais grupos de clientes parecem ter maior risco de churn?

## Etapas

1. Importação das bibliotecas
2. Carregamento dos dados
3. Exploração inicial
4. Limpeza e tratamento
5. Análise exploratória
6. Análise de churn
7. Análise por país
8. Visualizações
9. Insights finais
10. Conclusão

In [ ]:
# 1-2. Importação das bibliotecas e Carregando os dados

import pandas as pd

customer_info = pd.read_excel("../data/Bank_Churn_Messy.xlsx", sheet_name="Customer_Info")
account_info = pd.read_excel("../data/Bank_Churn_Messy.xlsx", sheet_name="Account_Info")

In [66]:
# 3. Exploração inicial

print(f"customer_info tem {customer_info.shape[0]} linhas e {customer_info.shape[1]} colunas.")
display(customer_info.head())
print(f"account_info tem {account_info.shape[0]} linhas e {account_info.shape[1]} colunas.")
display(account_info.head())


customer_info tem 10001 linhas e 8 colunas.


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
0,15634602,Hargrave,619,FRA,Female,42.0,2,€101348.88
1,15647311,Hill,608,Spain,Female,41.0,1,€112542.58
2,15619304,Onio,502,French,Female,42.0,8,€113931.57
3,15701354,Boni,699,FRA,Female,39.0,1,€93826.63
4,15737888,Mitchell,850,Spain,Female,43.0,2,€79084.1


account_info tem 10002 linhas e 7 colunas.


,CustomerId,Balance,NumOfProducts,HasCrCard,Tenure,IsActiveMember,Exited
0,15634602,€0.0,1,Yes,2,Yes,1
1,15634602,€0.0,1,Yes,2,Yes,1
2,15647311,€83807.86,1,Yes,1,Yes,0
3,15619304,€159660.8,3,No,8,No,1
4,15701354,€0.0,2,No,1,No,0


In [67]:
# Informações e estatísticas descritivas das tabelas

print("customer_info")
customer_info.info()
display(customer_info.describe().round(2))

print("account_info")
account_info.info()
display(account_info.describe().round(2))

customer_info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001 entries, 0 to 10000
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10001 non-null  int64  
 1   Surname          9998 non-null   object 
 2   CreditScore      10001 non-null  int64  
 3   Geography        10001 non-null  object 
 4   Gender           10001 non-null  object 
 5   Age              9998 non-null   float64
 6   Tenure           10001 non-null  int64  
 7   EstimatedSalary  10001 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 625.2+ KB


,CustomerId,CreditScore,Age,Tenure
count,10001.00,10001.00,9998.00,10001.00
mean,15690934.31,650.54,38.92,5.01
std,71935.31,96.66,10.49,2.89
min,15565701.00,350.00,18.00,0.00
25%,15628523.00,584.00,32.00,3.00
50%,15690733.00,652.00,37.00,5.00
75%,15753229.00,718.00,44.00,7.00
max,15815690.00,850.00,92.00,10.00


account_info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10002 entries, 0 to 10001
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CustomerId      10002 non-null  int64 
 1   Balance         10002 non-null  object
 2   NumOfProducts   10002 non-null  int64 
 3   HasCrCard       10002 non-null  object
 4   Tenure          10002 non-null  int64 
 5   IsActiveMember  10002 non-null  object
 6   Exited          10002 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 547.1+ KB


,CustomerId,NumOfProducts,Tenure,Exited
count,10002.00,10002.00,10002.00,10002.0
mean,15690928.68,1.53,5.01,0.2
std,71933.92,0.58,2.89,0.4
min,15565701.00,1.00,0.00,0.0
25%,15628524.75,1.00,3.00,0.0
50%,15690732.00,1.00,5.00,0.0
75%,15753225.50,2.00,7.00,0.0
max,15815690.00,4.00,10.00,1.0


In [68]:
# Padronizando os nomes das colunas
print(f"customer_info colunas antigas:\n {customer_info.columns}")
customer_info.columns = customer_info.columns.str.strip().str.lower().str.replace(" ","_")
print(f"customer_info colunas novas:\n {customer_info.columns}")

print(f"\naccount_info colunas antigas:\n {account_info.columns}")
account_info.columns = account_info.columns.str.strip().str.lower().str.replace(" ","_")
print(f"account_info colunas novas:\n {account_info.columns}")

customer_info colunas antigas:
 Index(['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age',
       'Tenure', 'EstimatedSalary'],
      dtype='object')
customer_info colunas novas:
 Index(['customerid', 'surname', 'creditscore', 'geography', 'gender', 'age',
       'tenure', 'estimatedsalary'],
      dtype='object')

account_info colunas antigas:
 Index(['CustomerId', 'Balance', 'NumOfProducts', 'HasCrCard', 'Tenure',
       'IsActiveMember', 'Exited'],
      dtype='object')
account_info colunas novas:
 Index(['customerid', 'balance', 'numofproducts', 'hascrcard', 'tenure',
       'isactivemember', 'exited'],
      dtype='object')


In [69]:
# 4. Limpeza e tratamento

# Somando duplicadas
print(f"Na tabela customer_info foram encontradas {customer_info.duplicated().sum()} linhas duplicadas.")
print(f"Na tabela account_info foram encontradas {account_info.duplicated().sum()} linhas duplicadas.")

Na tabela customer_info foram encontradas 1 linhas duplicadas.
Na tabela account_info foram encontradas 2 linhas duplicadas.


In [70]:
# Removendo duplicadas:

customer_info = customer_info.drop_duplicates()
print(f"Na tabela customer_info foram encontradas {customer_info.duplicated().sum()} linhas duplicadas.")

account_info = account_info.drop_duplicates()
print(f"Na tabela account_info foram encontradas {account_info.duplicated().sum()} linhas duplicadas.")

Na tabela customer_info foram encontradas 0 linhas duplicadas.
Na tabela account_info foram encontradas 0 linhas duplicadas.


In [71]:
# Identificando valores nulos
print("Tabela customer_info")
display(customer_info.isnull().sum())
print("Tabela account_info")
display(account_info.isnull().sum())

Tabela customer_info


customerid         0
surname            3
creditscore        0
geography          0
gender             0
age                3
tenure             0
estimatedsalary    0
dtype: int64

Tabela account_info


customerid        0
balance           0
numofproducts     0
hascrcard         0
tenure            0
isactivemember    0
exited            0
dtype: int64

In [72]:
# Foram encontrados apenas 3 valores nulos na coluna surname e 3 na age. 
# Como são valores pequenos, irei preencher na coluna idade com a mediana da coluna.

customer_info["age"] = customer_info["age"].fillna(customer_info["age"].median())
display(customer_info.isnull().sum())


customerid         0
surname            3
creditscore        0
geography          0
gender             0
age                0
tenure             0
estimatedsalary    0
dtype: int64

In [73]:
# E removemos a coluna surname por não possuir valor para nossa analise, sendo utilizada apenas para identificação dos clientes.
customer_info = customer_info.drop(columns=["surname"])
display(customer_info.isnull().sum())


customerid         0
creditscore        0
geography          0
gender             0
age                0
tenure             0
estimatedsalary    0
dtype: int64

In [74]:
# Unimos as tabelas para centralizar as informações dos clientes e facilitar a análise.
# Vou utilizar inner join para manter apenas clientes presentes nas duas tabelas, garantindo que a análise seja feita somente com registros que possuem informações completas.

df = customer_info.merge(account_info, on="customerid", how="inner")
print(f"O df tem {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head(10)

O df tem 10000 linhas e 13 colunas.


,customerid,creditscore,geography,gender,age,tenure_x,estimatedsalary,balance,numofproducts,hascrcard,tenure_y,isactivemember,exited
0,15634602,619,FRA,Female,42.0,2,€101348.88,€0.0,1,Yes,2,Yes,1
1,15647311,608,Spain,Female,41.0,1,€112542.58,€83807.86,1,Yes,1,Yes,0
2,15619304,502,French,Female,42.0,8,€113931.57,€159660.8,3,No,8,No,1
3,15701354,699,FRA,Female,39.0,1,€93826.63,€0.0,2,No,1,No,0
4,15737888,850,Spain,Female,43.0,2,€79084.1,€125510.82,1,Yes,2,Yes,0
5,15574012,645,Spain,Male,44.0,8,€149756.71,€113755.78,2,No,8,No,1
6,15592531,822,France,Male,50.0,7,€10062.8,€0.0,2,Yes,7,Yes,0
7,15656148,376,Germany,Female,29.0,4,€119346.88,€115046.74,4,No,4,No,1
8,15792365,501,French,Male,44.0,4,€74940.5,€142051.07,2,Yes,4,Yes,0
9,15592389,684,France,Male,27.0,2,€71725.73,€134603.88,1,Yes,2,Yes,0


In [75]:
# Analisando as informações da nova tabela após a união

print(df.duplicated().sum())
print(df.info())

0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       10000 non-null  int64  
 1   creditscore      10000 non-null  int64  
 2   geography        10000 non-null  object 
 3   gender           10000 non-null  object 
 4   age              10000 non-null  float64
 5   tenure_x         10000 non-null  int64  
 6   estimatedsalary  10000 non-null  object 
 7   balance          10000 non-null  object 
 8   numofproducts    10000 non-null  int64  
 9   hascrcard        10000 non-null  object 
 10  tenure_y         10000 non-null  int64  
 11  isactivemember   10000 non-null  object 
 12  exited           10000 non-null  int64  
dtypes: float64(1), int64(6), object(6)
memory usage: 1015.8+ KB
None


In [76]:
# Com a junção ficamos com duas colunas tenure (tenure_x e tenure_y), irei comparar as duas para conferir se os valores estão realmente iguais nas duas

(df["tenure_x"] == df["tenure_y"]).value_counts()

True    10000
Name: count, dtype: int64

In [77]:
# Como as duas são 100% iguais, irei criar uma nova coluna ternure e deletar as duas.

print(df.columns)
df["tenure"] = df["tenure_x"]

df = df.drop(columns=["tenure_x", "tenure_y"])
print(df.columns)

df.head(5)

Index(['customerid', 'creditscore', 'geography', 'gender', 'age', 'tenure_x',
       'estimatedsalary', 'balance', 'numofproducts', 'hascrcard', 'tenure_y',
       'isactivemember', 'exited'],
      dtype='object')
Index(['customerid', 'creditscore', 'geography', 'gender', 'age',
       'estimatedsalary', 'balance', 'numofproducts', 'hascrcard',
       'isactivemember', 'exited', 'tenure'],
      dtype='object')


,customerid,creditscore,geography,gender,age,estimatedsalary,balance,numofproducts,hascrcard,isactivemember,exited,tenure
0,15634602,619,FRA,Female,42.0,€101348.88,€0.0,1,Yes,Yes,1,2
1,15647311,608,Spain,Female,41.0,€112542.58,€83807.86,1,Yes,Yes,0,1
2,15619304,502,French,Female,42.0,€113931.57,€159660.8,3,No,No,1,8
3,15701354,699,FRA,Female,39.0,€93826.63,€0.0,2,No,No,0,1
4,15737888,850,Spain,Female,43.0,€79084.1,€125510.82,1,Yes,Yes,0,2


In [78]:
# Como vimos acima, as colunas 'EstimatedSalary' e 'Balance' estão como texto.
# Para fazer análises estatísticas, precisamos convertê-las para o tipo numérico.

df["estimatedsalary"] = pd.to_numeric(
    df["estimatedsalary"]
    .astype(str)
    .str.replace("€","", regex=False)
    .str.replace(",","", regex=False)
    .str.strip(),
    errors="coerce"
)

df["balance"] = pd.to_numeric(
    df["balance"]
    .astype(str)
    .str.replace("€","", regex=False)
    .str.replace(",","", regex=False)
    .str.strip(),
    errors="coerce"
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       10000 non-null  int64  
 1   creditscore      10000 non-null  int64  
 2   geography        10000 non-null  object 
 3   gender           10000 non-null  object 
 4   age              10000 non-null  float64
 5   estimatedsalary  10000 non-null  float64
 6   balance          10000 non-null  float64
 7   numofproducts    10000 non-null  int64  
 8   hascrcard        10000 non-null  object 
 9   isactivemember   10000 non-null  object 
 10  exited           10000 non-null  int64  
 11  tenure           10000 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 937.6+ KB


In [79]:
# Na coluna geography temos valores diferentes
df["geography"].value_counts()

geography
Germany    2509
Spain      2477
France     1741
French     1655
FRA        1618
Name: count, dtype: int64

In [80]:
# Irei padronizar os nomes para a analiser

df["geography"] = df["geography"].str.strip()

df["geography"] = df["geography"].replace(
    {
        "FRA": "France",
        "French": "France"
    }
)

df["geography"].value_counts()

geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [81]:
# Para finalizar, vou padronizar as colunas 'HasCrCard' e 'IsActiveMember' para 1 e 0 facilitando futuros cálculos.

df["hascrcard"] = df["hascrcard"].str.strip().replace(
    {
        "Yes": 1,
        "No": 0
    }
)

df["isactivemember"] = df["isactivemember"].str.strip().replace({
    "Yes": 1,
    "No": 0
})

C:\Users\conta\AppData\Local\Temp\ipykernel_133964\1378186630.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["hascrcard"] = df["hascrcard"].str.strip().replace(
C:\Users\conta\AppData\Local\Temp\ipykernel_133964\1378186630.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["isactivemember"] = df["isactivemember"].str.strip().replace({


In [82]:
# Criando uma nova coluna de churn com base na coluna exited para ficar mais facil a compreensão
df["churn"] = df["exited"]

# Contando a quantidade de cancelamentos
print(df["churn"].value_counts())
# Calculando a porcentagem de cancelamentos
print(df["churn"].value_counts(normalize=True))

churn
0    7963
1    2037
Name: count, dtype: int64
churn
0    0.7963
1    0.2037
Name: proportion, dtype: float64


Conferindo todas as alterações antes das análises

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       10000 non-null  int64  
 1   creditscore      10000 non-null  int64  
 2   geography        10000 non-null  object 
 3   gender           10000 non-null  object 
 4   age              10000 non-null  float64
 5   estimatedsalary  10000 non-null  float64
 6   balance          10000 non-null  float64
 7   numofproducts    10000 non-null  int64  
 8   hascrcard        10000 non-null  int64  
 9   isactivemember   10000 non-null  int64  
 10  exited           10000 non-null  int64  
 11  tenure           10000 non-null  int64  
 12  churn            10000 non-null  int64  
dtypes: float64(3), int64(8), object(2)
memory usage: 1015.8+ KB


In [84]:
print(df.isnull().sum())
df.head(10)

customerid         0
creditscore        0
geography          0
gender             0
age                0
estimatedsalary    0
balance            0
numofproducts      0
hascrcard          0
isactivemember     0
exited             0
tenure             0
churn              0
dtype: int64


,customerid,creditscore,geography,gender,age,estimatedsalary,balance,numofproducts,hascrcard,isactivemember,exited,tenure,churn
0,15634602,619,France,Female,42.0,101348.88,0.00,1,1,1,1,2,1
1,15647311,608,Spain,Female,41.0,112542.58,83807.86,1,1,1,0,1,0
2,15619304,502,France,Female,42.0,113931.57,159660.80,3,0,0,1,8,1
3,15701354,699,France,Female,39.0,93826.63,0.00,2,0,0,0,1,0
4,15737888,850,Spain,Female,43.0,79084.10,125510.82,1,1,1,0,2,0
5,15574012,645,Spain,Male,44.0,149756.71,113755.78,2,0,0,1,8,1
6,15592531,822,France,Male,50.0,10062.80,0.00,2,1,1,0,7,0
7,15656148,376,Germany,Female,29.0,119346.88,115046.74,4,0,0,1,4,1
8,15792365,501,France,Male,44.0,74940.50,142051.07,2,1,1,0,4,0
9,15592389,684,France,Male,27.0,71725.73,134603.88,1,1,1,0,2,0


In [85]:
# Agora vou exportar a base tratada para utilizá-la na análise das perguntas de negócio.

df.to_csv("../data/bank_churn_clear.csv")